In [1]:
# 10 Claim history: is there signal the GLM v3 is missing?
# Purpose: count past claims known at the start of each policy year, then check GLM v3 validation A/E by
# history group. A/E well above 1 for rows with past claims = risk the current model does not price.
# "Known" = reported before 1 Jan of the policy year, the same information a 2025 test row has (data end 2024-12-30).
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims

claims = load_claims()                                   # de-duplicated, repudiated removed, all perils
pol_tr = {l: load_policies(l, "train") for l in LINES}
pol_te = {l: load_policies(l, "test") for l in LINES}
all_pol = list(pol_tr.values()) + list(pol_te.values())


def claim_history(target_pol, all_pol, claims):
    """Per row_id: claims reported before 1 Jan of the policy year, with loss date in the previous
    1 or 3 years, on the same policy and on the same customer (all lines). Zero-net claims included."""
    t = target_pol[["row_id", "policy_id", "customer_id", "year"]].copy()
    t["cutoff"] = pd.to_datetime(t["year"].astype(str) + "-01-01")
    pmap = pd.concat([p[["policy_id", "customer_id"]] for p in all_pol]).drop_duplicates("policy_id")
    c = claims[["policy_id", "line", "loss_date", "report_date"]].merge(pmap, on="policy_id", how="left")
    out = t[["row_id"]].copy()
    for key, tag in [("policy_id", "pol"), ("customer_id", "cust")]:
        j = t.merge(c[[key, "loss_date", "report_date"]], on=key, how="inner")
        j = j[j.report_date < j.cutoff]
        for yrs in [1, 3]:
            w = j[j.loss_date >= j.cutoff - pd.DateOffset(years=yrs)]
            out = out.merge(w.groupby("row_id").size().rename(f"{tag}_claims_{yrs}y"), on="row_id", how="left")
    return out.fillna(0)


# GLM v3 validation predictions (09c, features dropped)
vp = pd.read_parquet("../results/feature_drop_valid_preds.parquet")
PRED = "GLM drop all candidates"


def ae_table(d, col, bins, labels):
    g = d.assign(grp=pd.cut(d[col], bins=bins, labels=labels, right=False)).groupby(["grp", "year"], observed=True)
    t = g.apply(lambda x: pd.Series({"weight_share": x.weight.sum(), "A/E": x.actual.sum() / x[PRED].sum(),
                                     "cost_per_weight": x.actual.sum() / x.weight.sum()}))
    t["weight_share"] = t.weight_share / t.groupby(level="year").weight_share.transform("sum")
    return t.unstack("year").round(3)


for line in LINES:
    p = pol_tr[line]
    h = claim_history(p, all_pol, claims)
    d = (vp[vp.line == line].merge(h, on="row_id", how="left")
         .merge(p[["row_id", "new_business"]], on="row_id", how="left"))
    d["other_lines_3y"] = d.cust_claims_3y - d.pol_claims_3y          # claims on the customer's other policies
    print(f"\n==================== {line} ====================")
    print("[same policy, claims in the previous year] GLM v3 validation A/E (> 1 = under-priced):")
    print(ae_table(d[d.new_business == 0], "pol_claims_1y", [0, 1, 2, np.inf], ["0", "1", "2+"]).to_string())
    print("\n[same policy, previous 3 years]")
    print(ae_table(d[d.new_business == 0], "pol_claims_3y", [0, 1, 2, 3, np.inf], ["0", "1", "2", "3+"]).to_string())
    print("\n[customer's OTHER policies (any line), previous 3 years]")
    print(ae_table(d, "other_lines_3y", [0, 1, 2, np.inf], ["0", "1", "2+"]).to_string())
    print("\n[new business vs renewal]")
    print(ae_table(d, "new_business", [0, 1, 2], ["renewal", "new business"]).to_string())

    # compare with the self-reported prior_claims_5y already in the model
    cust = pd.read_parquet("../data/customers.parquet")[["customer_id", "prior_claims_5y"]] \
        if "prior_claims_5y" not in p else None
    pc = p[["row_id", "customer_id"]].merge(cust, on="customer_id", how="left") if cust is not None \
        else p[["row_id", "prior_claims_5y"]]
    d2 = d.merge(pc[["row_id", "prior_claims_5y"]], on="row_id", how="left")
    print("\n[self-reported prior_claims_5y (already a model feature)]")
    print(ae_table(d2, "prior_claims_5y", [0, 1, 2, np.inf], ["0", "1", "2+"]).to_string())
    corr = d2[["pol_claims_3y", "cust_claims_3y", "prior_claims_5y"]].corr(method="spearman").round(2)
    print("\nSpearman correlation, built history vs self-reported:\n", corr.to_string())

    # test rows: is the history distribution similar to 2024?
    ht = claim_history(pol_te[line], all_pol, claims)
    h24 = h[h.row_id.isin(p.row_id[p.year == 2024])]
    print(f"\nshare with >= 1 claim on the same policy in the previous year: "
          f"train 2024 {(h24.pol_claims_1y > 0).mean():.1%} | test 2025 {(ht.pol_claims_1y > 0).mean():.1%}")


==================== motor ====================
[same policy, claims in the previous year] GLM v3 validation A/E (> 1 = under-priced):
     weight_share           A/E        cost_per_weight          
year         2023   2024   2023   2024            2023      2024
grp                                                             
0           0.890  0.894  0.966  0.976         452.728   452.578
1           0.097  0.095  1.223  1.242         819.661   808.284
2+          0.012  0.011  1.813  1.870        1719.090  1649.867

[same policy, previous 3 years]
     weight_share           A/E        cost_per_weight          
year         2023   2024   2023   2024            2023      2024
grp                                                             
0           0.812  0.762  0.949  0.903         433.498   406.488
1           0.152  0.179  1.124  1.231         680.421   681.614
2           0.029  0.042  1.380  1.416        1080.721   963.108
3+          0.008  0.017  1.971  1.824        2189.